In [ ]:
#| hide
from yasi.core import *

# yasi

> **y**et **a**nother **s**olve**i**t — Dialog Engineering in JupyterLab

## Background

What if you could have a more fluid, interactive conversation with AI? Enter **Dialog Engineering**: instead of one-shot prompts you construct and edit a dialog with the AI, and the whole dialog gets sent on every turn.

> Unlike prompt engineering where you're just creating a single sentence or paragraph or whatever, that's actually part of a whole back and forth dialog. All of the previous steps get sent to the AI model as well, not just the prompt. And they all greatly influence how it responds. And how it responds influences you as to what you then add to the dialog. - Jeremy Howard, from [the MAD Podcast 34:42](https://www.youtube.com/watch?v=MbHL0uvKYbE&t=34h42s)

yasi turns a JupyterLab notebook into that dialog. It is a DIY take on [solveit](https://solveit.fast.ai/) from answer.ai: your notebook *is* the conversation — notes, code, and outputs included — and you can edit any part of it before the next turn.

## Installation

``` sh
$ pip install git+https://github.com/Jack-Byte/yasi.git
```

or from [pypi](https://pypi.org/project/yasi/)

``` sh
$ pip install ipy-yasi
```

## Quickstart

yasi talks to any OpenAI-compatible API:

``` python
from yasi.core import JupyterChat

# OpenRouter (recommended: one key, many models, per-token pricing shown in the panel)
jc = JupyterChat(openai_base_url="https://openrouter.ai/api/v1", api_key=None)  # or env var OPENAI_API_KEY

# Anthropic directly (note the trailing /v1/)
jc = JupyterChat(openai_base_url="https://api.anthropic.com/v1/", api_key=os.environ["ANTHROPIC_API_KEY"])

# Local Ollama
jc = JupyterChat(openai_base_url="http://localhost:11434/v1", api_key="ollama")
```

This adds a *Yasi* panel on the right (model picker, request parameters, token/cost display) and three notebook toolbar buttons: add user message, add assistant message, send dialog.

## How the dialog is built

When you send the dialog (toolbar button or `%%prompt`), yasi reads the saved notebook and turns it into messages:

- **Everything visible goes to the model** (solveit-style): untagged markdown cells become `user` messages; code cells become `user` messages containing their source *and outputs* (tracebacks cleaned, images skipped, long output truncated via `max_output_len`).
- **Role tags override**: cells starting with `#| chat_system`, `#| chat_user` or `#| chat_assistant` are sent with that role. Responses inserted by yasi are tagged `#| chat_assistant` automatically.
- **Hiding**: give a cell the `yasi-hide` tag (Property Inspector) or a `#| yasi-hide` source line and it is never sent.
- **Classic mode**: `JupyterChat(all_cells=False)` restores the original tagged-only behavior.

## Prompt cells

The `%%prompt` cell magic (registered automatically) gives you solveit's run-a-prompt flow:

``` python
%%prompt
What is money? Explain it like my mate from Oz.
```

Running the cell sends everything *above* it plus the prompt. The response streams live into the prompt cell's output while it is generated, and is then inserted as a tagged markdown cell below. Cells below the prompt are not sent.

## More options

``` python
jc = JupyterChat(
    openai_base_url="https://openrouter.ai/api/v1",
    model="meta-llama/llama-3.1-8b-instruct",
    system_prompt=None,     # disable or replace the default teaching prompt
    all_cells=True,        # False = classic tagged-only mode
    max_output_len=2000,   # truncate code cell outputs
    hide_tag="yasi-hide",  # the hide-from-model tag
)

jc.usage  # {'prompt_tokens': ..., 'completion_tokens': ..., 'total_tokens': ...}
```

By default yasi uses a solveit-style *teaching* system prompt: the model acts as a pair-programming partner that guides with hints and small steps instead of dumping finished code (one concise solution when code is really needed). Pass your own `system_prompt`, or `None` to disable it.

Token usage of the session is shown in the Yasi panel after every exchange; with OpenRouter the estimated cost is shown too.

Anthropic quirks (temperature range 0-1, required `max_tokens`, unsupported `presence_penalty`) are handled automatically.

## Try it online

You can try it with Binder [![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/Jack-Byte/yasi/HEAD?urlpath=%2Fdoc%2Ftree%2F%2Fexamples%2Fmoney_and_kangaroos.ipynb)

For more information see the following articles:

- [Dialog Engineering](https://www.linkedin.com/pulse/dialog-engineering-tobias-klings-jq34f)
- [Introducing Dialog Engineering with yasi](https://www.linkedin.com/pulse/boost-your-ai-interactions-tobias-klings-bcnqe)